In [5]:
import numpy as np
from scipy.special import softmax
import pandas as pd
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import ColumnDataSource
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from bokeh.transform import factor_cmap
output_notebook()

Loading BokehJS ...

In [6]:
import pandas as pd

col_names = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"
]

df = pd.read_csv(
    "census+income/adult.data",
    header=None,
    names=col_names,
    sep=r",\s*",
    engine="python",
    na_values="?"
)

# Keep fnlwgt out of model features for this experiment.
df = df.drop(columns=["fnlwgt"])

# Binary target: >50K -> 1, <=50K -> 0
df["income"] = df["income"].astype(str).str.replace(".", "", regex=False).str.strip()
y = (df["income"] == ">50K").astype(int)
X = df.drop(columns=["income"])

print("Rows:", len(df))
print("Positive class rate:", y.mean())
print(X.head(2))

Rows: 32561
Positive class rate: 0.2408095574460244
   age         workclass  education  education_num      marital_status  \
0   39         State-gov  Bachelors             13       Never-married   
1   50  Self-emp-not-inc  Bachelors             13  Married-civ-spouse   

        occupation   relationship   race   sex  capital_gain  capital_loss  \
0     Adm-clerical  Not-in-family  White  Male          2174             0   
1  Exec-managerial        Husband  White  Male             0             0   

   hours_per_week native_country  
0              40  United-States  
1              13  United-States  


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "age", "education_num", "capital_gain", "capital_loss", "hours_per_week"
]

categorical_features = [
    "workclass", "education", "marital_status", "occupation",
    "relationship", "race", "sex", "native_country"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=2000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=11, stratify=y
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8491585800270237

Confusion matrix:
 [[5750  431]
 [ 797 1163]]

Classification report:
               precision    recall  f1-score   support

           0       0.88      0.93      0.90      6181
           1       0.73      0.59      0.65      1960

    accuracy                           0.85      8141
   macro avg       0.80      0.76      0.78      8141
weighted avg       0.84      0.85      0.84      8141



In [10]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

# Export matrix for cross-model comparison plots.
os.makedirs("exports", exist_ok=True)
lr_cm_path = os.path.join("exports", "logistic_regression_confusion_matrix.csv")
np.savetxt(lr_cm_path, cm, fmt="%d", delimiter=",")
print(f"Saved Logistic Regression confusion matrix to: {lr_cm_path}")

Saved Logistic Regression confusion matrix to: exports\logistic_regression_confusion_matrix.csv
